In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import os

try: 
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.xla_multiprocessing as xmp
except: pass

from utils.utils_clf import *
from clf_models import load_model

from Diffusion.Diff_models import create_diff

from Diffusion.gaussian_diffusion import (
    GaussianDiffusion,
    get_named_beta_schedule,
    ModelMeanType,
    ModelVarType,
    LossType)
import argparse

from purify import *

device = xm.xla_device() if 'xm' in globals() else torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [24]:
os.environ['XRT_TPU_CONFIG'] = 'localservice;0;localhost:51011'

# ======== arg parser =================================================
parser = argparse.ArgumentParser(description='PyTorch Poison Attack')

### Setup Arguments ###
parser.add_argument('--remote_user', type=str, help='username for the remote server (TPU only, else pass in full directory args below)')
# parser.add_argument('--num_proc', type=int, default=8, help='number of processes for TPU')
parser.add_argument('--device_type', default='xla', type=str, choices=['xla','cuda','cpu','mps'],help='device type to use')
parser.add_argument('--seed', default=11, type=int,help='seed for reproducibility')
parser.add_argument('--verbose','--v', default=False, action='store_true',help='print out additional information when running')
parser.add_argument('--data_dir', default='/home/data/', type=str, help='path to the data directory')
parser.add_argument('--num_proc', type=int, default=1, help='number of processes for TPU')

### Experiment Arguments ###
parser.add_argument('--dataset', default='cifar10', type=str, choices=['cifar10','cinic10','stl10','stl10_64','tinyimagenet'],help='dataset to use')

### Purification Arguments ###

parser.add_argument('--purify_reps', default=1, type=int, help='number of purification repetitions (when using both EBM and Diffusion)')

# EBM Arguments 
args_ebm = parser.add_argument_group('EBM')
args_ebm.add_argument('--ebm_model', default='EBMSNGAN32', type=none_or_str, choices=[None,'SuperLightEBM','LightEBM','EBM','EBMSNGAN32','EBMSNGAN128','EBMSNGAN256'],help='type of EBM model to use')
args_ebm.add_argument('--ebm_name', default='ebm_cifar10_45k', type=str_or_str_list, help='path to the EBM model including train dataset')
args_ebm.add_argument('--ebm_nf', default=128, type=int_or_int_list,  help='number of filters for the ebm model')
args_ebm.add_argument('--ebm_lang_steps', default=150, type=int_or_int_list, help='number of langevin steps')
args_ebm.add_argument('--ebm_lang_temp', default=1e-4, type=float_or_float_list, help='langevin temperature')

# Diffusion Arguments
args_diff = parser.add_argument_group('Diffusion')
args_diff.add_argument('--diff_model', default='DDPM_UNET_EBM', type=none_or_str, choices=[None, 'DDPM_UNET','DDPM_UNET_EBM'],help='type of diffusion model to use')
args_diff.add_argument('--diff_name', default='mcmc_steps1600_bank_fixer_cosine', type=str_or_str_list, help='path to the diffusion model')
args_diff.add_argument('--diff_nf', default=128, type=int_or_int_list,  help='number of filters for the unet model')
args_diff.add_argument('--diff_train_steps', default=1000, type=int_or_int_list, help='training t-steps for diffuion model')
args_diff.add_argument('--diff_output', default='epsilon', type=str, choices=['epsilon','start_x'],  help='diffusion model output')
args_diff.add_argument('--diff_schedule', default='cosine', type=str, choices=['linear','cosine'], help='t schedule')
args_diff.add_argument('--diff_purify_steps', default=10, type=int_or_int_list,  help='number of purify t-steps for the unconditional diffuion model')
args_diff.add_argument('--diff_eta', default=0, type=int_or_int_list,  help='ddpm 1 or ddim 0 for the sampling of the 1000 tstep fixer')
    

### Poison Arguments ###
parser.add_argument('--poison_type', default=None, type=str, choices=['Narcissus', 'Gradient_Matching','BullseyePolytope','BullseyePolytope_Bench'],help='type of poison to generate')
parser.add_argument('--poison_mode', default='from_scratch', type=str, choices=['from_scratch','transfer'],help='mode of attack')
parser.add_argument('--noise_sz_narcissus', default=32, type=int, help='size of the noise trigger for Narcissus')
parser.add_argument('--noise_eps_narcissus', default=8, type=int, help='epsilon for the noise trigger for Narcissus')
parser.add_argument('--num_images_narcissus', default=500, type=int, help='number of poisoned images generated')
parser.add_argument('--random_imgs_narcissus', default=False, action='store_true', help='use random images for narcissus')
parser.add_argument('--iters_bp', default=800, type=int,help='iterations for making poison')
parser.add_argument('--num_images_bp', default=50, type=int,help='number of poisoned images generated')
parser.add_argument('--net_repeat_bp', default=1, type=int, help='number of times to repeat the network for methods BP-1, BP-3, BP-5')
parser.add_argument('--num_per_class_bp', default=50, type=int, help='num of samples per class for re-training, or the poison dataset')

# Parse the arguments
args = parser.parse_args(['--remote_user','sunaybhat'])

if args.remote_user is not None:
    args.data_dir = args.data_dir.replace('/home',f'/home/{args.remote_user}')

In [27]:
# Get diff and ebm model paths
if args.ebm_model is not None: ebm_path = os.path.join(args.data_dir,'models',args.ebm_model,args.ebm_name+'.pt')
else: ebm_path = None
if args.diff_model is not None: diff_path = os.path.join(args.data_dir,'models',args.diff_model,args.diff_name+'.pt')
else: diff_path = None

# Create the PureDefense object
PurifyClass = PureDefense(device,args.device_type,
                        ebm_type=args.ebm_model,ebm_path=ebm_path,ebm_nf=args.ebm_nf,
                        diff_type=args.diff_model,diff_path=diff_path,diff_nf=args.diff_nf,
                        diff_schedule=args.diff_schedule,
                        diff_train_steps=args.diff_train_steps,diff_output=args.diff_output,
                        img_sz=32,verbose=args.verbose)

1000


In [3]:
diff_model = create_diff('DDPM_UNET_EBM',32,128)

# Print number of parameters
num_params = sum(p.numel() for p in diff_model.parameters() if p.requires_grad)
print(f'Number of parameters: {num_params}')

Number of parameters: 49066243


In [4]:
betas = get_named_beta_schedule(diff_schedule,diff_steps) 

NameError: name 'diff_schedule' is not defined

In [5]:
import torch
import torch.nn as nn

class SmallUNet(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim=32, num_res_blocks=2, nf=16):
        super(SmallUNet, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.time_emb_dim = time_emb_dim
        self.num_res_blocks = num_res_blocks
        self.nf = nf

        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, nf * 4),
            nn.ReLU(),
            nn.Linear(nf * 4, nf * 4),
        )

        self.input_conv = nn.Conv2d(in_channels, nf, kernel_size=3, padding=1)

        self.down1 = self._make_res_block(nf, nf * 2, stride=2)
        self.down2 = self._make_res_block(nf * 2, nf * 4, stride=2)

        self.mid_block = nn.ModuleList([
            self._make_res_block(nf * 4, nf * 4) for _ in range(num_res_blocks)
        ])

        self.up1 = self._make_res_block(nf * 8, nf * 2, stride=2, transpose=True)
        self.up2 = self._make_res_block(nf * 4, nf, stride=2, transpose=True)

        self.output_conv = nn.Conv2d(nf * 2, out_channels, kernel_size=1)

    def _make_res_block(self, in_channels, out_channels, stride=1, transpose=False):
        if transpose:
            conv = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, output_padding=1)
        else:
            conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)

        return nn.Sequential(
            conv,
            nn.GroupNorm(8, out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        x = self.input_conv(x)
        skip_connections = []

        x = self.down1(x)
        skip_connections.append(x)
        x = self.down2(x)

        for block in self.mid_block:
            x = block(x)

        x = torch.cat([x, skip_connections.pop()], dim=1)
        x = self.up1(x)
        x = torch.cat([x, skip_connections.pop()], dim=1)
        x = self.up2(x)

        x = self.output_conv(x)

        return x

In [7]:
model = SmallUNet(3, 3, time_emb_dim=32, num_res_blocks=2, nf=32)

# Print number of parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Number of parameters: {num_params}')

Number of parameters: 1121795


In [13]:
import torch
import torch.nn as nn

class SmallUNetAttention(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim=64, num_res_blocks=2, nf=32, num_heads=4):
        super(SmallUNetAttention, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.time_emb_dim = time_emb_dim
        self.num_res_blocks = num_res_blocks
        self.nf = nf
        self.num_heads = num_heads

        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, nf * 4),
            nn.ReLU(),
            nn.Linear(nf * 4, nf * 4),
        )

        self.input_conv = nn.Conv2d(in_channels, nf, kernel_size=3, padding=1)

        self.down1 = self._make_res_block(nf, nf * 2, stride=2)
        self.attn1 = AttentionBlock(nf * 2, num_heads)
        self.down2 = self._make_res_block(nf * 2, nf * 4, stride=2)
        self.attn2 = AttentionBlock(nf * 4, num_heads)

        self.mid_block = nn.ModuleList([
            self._make_res_block(nf * 4, nf * 4) for _ in range(num_res_blocks)
        ])

        self.up1 = self._make_res_block(nf * 8, nf * 2, stride=2, transpose=True)
        self.attn3 = AttentionBlock(nf * 2, num_heads)
        self.up2 = self._make_res_block(nf * 4, nf, stride=2, transpose=True)
        self.attn4 = AttentionBlock(nf, num_heads)

        self.output_conv = nn.Conv2d(nf * 2, out_channels, kernel_size=1)

    def _make_res_block(self, in_channels, out_channels, stride=1, transpose=False):
        if transpose:
            conv = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, output_padding=1)
        else:
            conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)

        return nn.Sequential(
            conv,
            nn.GroupNorm(8, out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        x = self.input_conv(x)
        skip_connections = []

        x = self.down1(x)
        x = self.attn1(x)
        skip_connections.append(x)
        x = self.down2(x)
        x = self.attn2(x)

        for block in self.mid_block:
            x = block(x)

        x = torch.cat([x, skip_connections.pop()], dim=1)
        x = self.up1(x)
        x = self.attn3(x)
        x = torch.cat([x, skip_connections.pop()], dim=1)
        x = self.up2(x)
        x = self.attn4(x)

        x = self.output_conv(x)

        return x

class AttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=4):
        super(AttentionBlock, self).__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.q = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.k = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.v = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.proj_out = nn.Conv2d(channels, channels, kernel_size=1)
        self.num_heads = num_heads

    def forward(self, x):
        b, c, h, w = x.shape
        q = self.q(x).view(b, self.num_heads, c // self.num_heads, h * w).permute(0, 1, 3, 2)
        k = self.k(x).view(b, self.num_heads, c // self.num_heads, h * w)
        v = self.v(x).view(b, self.num_heads, c // self.num_heads, h * w).permute(0, 1, 3, 2)

        attn = torch.matmul(q, k) * (c ** (-0.5))
        attn = torch.softmax(attn, dim=-1)

        out = torch.matmul(attn, v)
        out = out.permute(0, 1, 3, 2).contiguous().view(b, c, h, w)
        out = self.proj_out(out)

        return out

In [14]:
model = SmallUNetAttention(3, 3, time_emb_dim=32, num_res_blocks=2, nf=32, num_heads=4)

# Print number of parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Number of parameters: {num_params}')

Number of parameters: 1225059


In [15]:
input = torch.randn(1, 3, 32, 32)

output = model(input, torch.randn(1, 32))

print(output.shape)

RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 8 but got size 16 for tensor number 1 in the list.